# Generate (n, k, m, P) Training Data with LP-computed m-Heights

Generates 100,000 random samples and computes their exact m-heights using the LP-based algorithm.

- **n** = 9 (fixed)
- **k** ∈ {4, 5, 6}
- **m** ∈ {2, ..., n−k}
- **P** is a k × (n−k) matrix of integers drawn from **Normal(0, σ)** where σ is tuned per (k,m) group to match the professor's reference data distribution. The professor uses Gaussian distributions (per Jiang & Zuo, NVMW 2024). Per-group σ values are estimated from the reference data via σ = mean|P| / √(2/π).

In [19]:
import numpy as np
from scipy.optimize import linprog
from itertools import combinations, product
from multiprocessing import Pool, cpu_count
import pickle
import time
import sys

## 1. Generate 100,000 random (n, k, m, P) samples

We draw uniformly across the 9 valid (k, m) combinations using a **mixed generation strategy**:

- **80% Gaussian P**: entries sampled from Normal(0, σ_km), rounded to integers, clipped to [−100, 100]. Produces the full right-skewed m-height distribution (median ~300, tail to millions).
- **20% Punctured APC P**: columns are permutations of a Gaussian base vector — the *Analog Permutation Code* construction from Jiang & Zuo NVMW 2024. These produce the rare low-m-height region (heights near 2–50) that random Gaussian sampling almost never reaches.

**Why mix?** The professor's training data has m-heights from 2.0 to 6.45M (median 307). Pure random Gaussian never reaches the low end; pure APC/search never reaches the high tail. The 80/20 mix reproduces both regions.

**Per-group σ** is estimated from reference data as σ = mean|P|_ref / √(2/π):

| (k,m)  | σ  | Ref mean\|P\| |
|--------|----|--------------|
| (4, 2) | 37 | 29.89        |
| (4, 3) | 38 | 30.18        |
| (4, 4) | 40 | 31.88        |
| (4, 5) | 51 | 40.41        |
| (5, 2) | 35 | 27.95        |
| (5, 3) | 37 | 29.25        |
| (5, 4) | 46 | 36.95        |
| (6, 2) | 29 | 23.05        |
| (6, 3) | 40 | 32.22        |

In [ ]:
N_SAMPLES = 100_000
N = 9
rng = np.random.default_rng(seed=2026)

# All valid (k, m) combinations
configs = []
for k in [4, 5, 6]:
    for m in range(2, N - k + 1):  # m in {2, ..., n-k}
        configs.append((k, m))

print(f"Valid (k, m) combinations ({len(configs)} total):")
for k, m in configs:
    print(f"  k={k}, m={m}  -> P shape: ({k}, {N-k})")

# Per-(k,m) Normal sigma values — estimated from reference data as σ = mean|P| / sqrt(2/π)
sigma_map = {
    (4, 2): 37,
    (4, 3): 38,
    (4, 4): 40,
    (4, 5): 51,
    (5, 2): 35,
    (5, 3): 37,
    (5, 4): 46,
    (6, 2): 29,
    (6, 3): 40,
}

def min_distance(P_mat, k, n):
    """Compute minimum distance d of the code [I_k | P] via parity check matrix.
    d = smallest number of linearly dependent columns of H = [-P^T | I_{n-k}]."""
    r = n - k
    H = np.hstack([-P_mat.T, np.eye(r)])
    for t in range(1, n + 1):
        for cols in combinations(range(n), t):
            sub = H[:, list(cols)]
            if np.linalg.matrix_rank(sub, tol=1e-8) < min(t, r):
                return t
    return n + 1

def gen_gaussian_P(k, n, rng_obj, sigma, clip_value=100):
    """Sample P from Normal(0, sigma), rounded to integers, clipped to [-clip, clip]."""
    r = n - k
    return np.clip(
        np.round(rng_obj.normal(0, sigma, size=(k, r))), -clip_value, clip_value
    ).astype(int)

def gen_apc_P(k, n, rng_obj, sigma, clip_value=100):
    """Punctured Analog Permutation Code (Jiang & Zuo NVMW 2024):
    Columns of P are distinct permutations of a single Gaussian base vector.
    For n != k! we puncture: select r = n-k columns from the k! permutations.
    These produce structured codewords where entries repeat with permuted signs/order,
    giving small max/min-entry ratios -> low m-heights.
    """
    from itertools import permutations as _perms
    r = n - k
    base = rng_obj.normal(0, sigma, size=k)
    all_perms = list({tuple(p) for p in _perms(base)})
    rng_obj.shuffle(all_perms)
    cols = [np.array(all_perms[i % len(all_perms)]) for i in range(r)]
    return np.clip(np.round(np.stack(cols, axis=1)), -clip_value, clip_value).astype(int)

# 80% Gaussian + 20% punctured APC — mirrors the paper's genetic programming initial population
# (random Gaussian codes + random Analog Permutation Codes and their punctured codes)
APC_FRACTION = 0.20

samples = []
rejected = 0
for i in range(N_SAMPLES):
    while True:
        k, m = configs[rng.integers(len(configs))]
        sigma = sigma_map[(k, m)]
        if rng.random() < APC_FRACTION:
            P = gen_apc_P(k, N, rng, sigma)
        else:
            P = gen_gaussian_P(k, N, rng, sigma)
        d = min_distance(P, k, N)
        if m < d:
            samples.append([N, k, m, P])
            break
        rejected += 1
    if (i + 1) % 10_000 == 0:
        print(f"  Generated {i+1}/{N_SAMPLES} samples ({rejected} rejected so far)")

# Verify distribution
from collections import Counter
km_counts = Counter((s[1], s[2]) for s in samples)
print(f"\nGenerated {len(samples)} samples ({rejected} rejected for m >= d). Distribution:")
for key in sorted(km_counts.keys()):
    print(f"  k={key[0]}, m={key[1]}: {km_counts[key]} samples")

s = samples[0]
print(f"\nSample 0: n={s[0]}, k={s[1]}, m={s[2]}, P shape={s[3].shape}, "
      f"P range=[{s[3].min()}, {s[3].max()}]")

P_flat = np.concatenate([s[3].flatten() for s in samples]).astype(float)
print(f"\nP distribution check:")
print(f"  std={P_flat.std():.1f}  (reference: 40.8)")
print(f"  mean|P|={np.mean(np.abs(P_flat)):.1f}  (reference: 30.1)")
print(f"  |P|<=10: {(np.abs(P_flat)<=10).mean()*100:.1f}%  (reference: 35.2%)")

## 2. LP-based m-Height computation

Exact LP algorithm from the project spec (Theorem 1). Each sample is solved independently, so we parallelize across CPU cores.

In [21]:


def compute_m_height(n, k, m, P):
    """
    Compute m-height using the simplified LP algorithm from:
    Roth et al., "On the Height Profile of Analog Error-Correcting Codes"
    (Theorem 1 + Figure 1).

    For each m-subset S of [n) (the "free" positions that can be large):
      - Constrain entries at positions S_bar to [-1, 1]:  -1 <= u·g_j <= 1 for j in S_bar
      - For each i in S, maximize u·g_i

    Total LPs: m * C(n, m).
    """
    G = np.hstack([np.eye(k), P])
    g_cols = G.T  # g_cols[j] = j-th column of G (length k)

    h = 0.0
    indices = list(range(n))

    # Enumerate all m-subsets S of [n) — these are the m "free" positions
    for S in combinations(indices, m):
        S_set = set(S)
        S_bar = [j for j in indices if j not in S_set]  # complement, size n-m

        # Constraints on S_bar: -1 <= u · g_j <= 1 for all j in S_bar
        G_Sbar = np.array([g_cols[j] for j in S_bar])  # (n-m, k)
        A_ub = np.vstack([G_Sbar, -G_Sbar])             # (2(n-m), k)
        b_ub = np.ones(2 * (n - m))

        # For each i in S (the free positions), maximize u · g_i
        for i in S:
            c_obj = -g_cols[i]  # negate for minimization
            result = linprog(c_obj, A_ub=A_ub, b_ub=b_ub,
                             bounds=[(None, None)] * k,
                             method='highs',
                             options={'presolve': True,
                                      'dual_feasibility_tolerance': 1e-8,
                                      'primal_feasibility_tolerance': 1e-8})
            if result.success:
                val = -result.fun
                if val > h:
                    h = val
            elif result.status == 3:  # unbounded
                return float('inf')

    return h


def _worker(args):
    """Worker function for multiprocessing."""
    idx, n, k, m, P = args
    height = compute_m_height(n, k, m, P)
    return idx, height


def _search_score_worker(args):
    """Score one candidate generator matrix across all target m values."""
    idx, G, n, k, target_ms = args
    scores = evaluate_G_multi_m(G, n, k, target_ms)
    scalar = float(np.mean([scores[m] for m in target_ms]))
    return idx, {"G": G, "scores": scores, "scalar": scalar}

print(f"LP solver ready (new algorithm). Available CPUs: {cpu_count()}")

# Quick comparison of LP counts: old vs new
from math import comb
print("\nLP count comparison (n=9):")
print(f"{'k':>3} {'m':>3} | {'Old algorithm':>14} | {'New algorithm':>14} | {'Speedup':>8}")
print("-" * 55)
for k_val in [4, 5, 6]:
    for m_val in range(2, 9 - k_val + 1):
        old = 9 * 8 * comb(7, m_val - 1) * (2 ** m_val)
        new = m_val * comb(9, m_val)
        print(f"{k_val:>3} {m_val:>3} | {old:>14,} | {new:>14,} | {old/new:>7.1f}x")


LP solver ready (new algorithm). Available CPUs: 32

LP count comparison (n=9):
  k   m |  Old algorithm |  New algorithm |  Speedup
-------------------------------------------------------
  4   2 |          2,016 |             72 |    28.0x
  4   3 |         12,096 |            252 |    48.0x
  4   4 |         40,320 |            504 |    80.0x
  4   5 |         80,640 |            630 |   128.0x
  5   2 |          2,016 |             72 |    28.0x
  5   3 |         12,096 |            252 |    48.0x
  5   4 |         40,320 |            504 |    80.0x
  6   2 |          2,016 |             72 |    28.0x
  6   3 |         12,096 |            252 |    48.0x


## 3. Compute m-heights in parallel

In [22]:
# Prepare work items
work = [(i, s[0], s[1], s[2], s[3]) for i, s in enumerate(samples)]

n_workers = 31
print(f"Computing m-heights for {len(work)} samples using {n_workers} workers...")
sys.stdout.flush()

m_heights = [None] * len(samples)
start = time.time()
done = 0

with Pool(n_workers) as pool:
    for idx, height in pool.imap_unordered(_worker, work, chunksize=10):
        m_heights[idx] = height
        done += 1
        if done % 200 == 0 or done == len(work):
            elapsed = time.time() - start
            rate = done / elapsed
            eta = (len(work) - done) / rate if rate > 0 else 0
            print(f"  [{done}/{len(work)}] {elapsed:.0f}s elapsed, "
                  f"{rate:.1f} samples/s, ETA {eta:.0f}s")
            sys.stdout.flush()

elapsed = time.time() - start
print(f"\nDone! {len(m_heights)} m-heights computed in {elapsed:.1f}s")
print(f"Sample heights (first 10): {m_heights[:10]}")

Computing m-heights for 100000 samples using 31 workers...
  [200/100000] 2s elapsed, 98.1 samples/s, ETA 1018s
  [400/100000] 3s elapsed, 123.6 samples/s, ETA 806s
  [600/100000] 4s elapsed, 141.0 samples/s, ETA 705s
  [800/100000] 5s elapsed, 150.9 samples/s, ETA 657s
  [1000/100000] 6s elapsed, 154.0 samples/s, ETA 643s
  [1200/100000] 7s elapsed, 164.3 samples/s, ETA 601s
  [1400/100000] 8s elapsed, 167.1 samples/s, ETA 590s
  [1600/100000] 10s elapsed, 166.9 samples/s, ETA 590s
  [1800/100000] 11s elapsed, 169.8 samples/s, ETA 578s
  [2000/100000] 12s elapsed, 168.8 samples/s, ETA 580s
  [2200/100000] 13s elapsed, 174.0 samples/s, ETA 562s
  [2400/100000] 14s elapsed, 174.7 samples/s, ETA 559s
  [2600/100000] 15s elapsed, 176.2 samples/s, ETA 553s
  [2800/100000] 16s elapsed, 177.8 samples/s, ETA 547s
  [3000/100000] 17s elapsed, 178.4 samples/s, ETA 544s
  [3200/100000] 18s elapsed, 177.8 samples/s, ETA 544s
  [3400/100000] 19s elapsed, 181.9 samples/s, ETA 531s
  [3600/100000] 2

## 4. Save to pickle files

In [23]:
out_dir = '/workspace/Homework/Project1'

with open(f'{out_dir}/Generated_n_k_m_P', 'wb') as f:
    pickle.dump(samples, f)

with open(f'{out_dir}/Generated_m-heights', 'wb') as f:
    pickle.dump(m_heights, f)

print(f"Saved {len(samples)} samples to Generated_n_k_m_P")
print(f"Saved {len(m_heights)} m-heights to Generated_m-heights")

# Verify by reloading
with open(f'{out_dir}/Generated_n_k_m_P', 'rb') as f:
    check_data = pickle.load(f)
with open(f'{out_dir}/Generated_m-heights', 'rb') as f:
    check_heights = pickle.load(f)
print(f"Verification: {len(check_data)} samples, {len(check_heights)} heights")
print(f"Heights range: [{min(check_heights):.6f}, {max(check_heights):.6f}]")

Saved 100000 samples to Generated_n_k_m_P
Saved 100000 m-heights to Generated_m-heights
Verification: 100000 samples, 100000 heights
Heights range: [11.760845, 49601671.999587]


## 5. Diverse code search: low-m-height enrichment + random scatter

Generates 10,000 additional (n, k, m, P) samples with a **two-part strategy per group**:

- **30% from evolutionary search** (low-m-height enrichment): runs the paper's genetic programming loop to find codes near the theoretical minimum. These cover the rare low-m-height region (heights 2–50) that random sampling almost never reaches.
- **70% random scatter** (full-distribution coverage): random Gaussian + APC P matrices computed in parallel. These restore the right-skewed tail (heights up to millions) that pure search squashes.

**Why both?** The professor's data spans [2, 6.45M] with median 307. Pure search collapses the range to [8, 5186]. Pure random misses the low end. The mix reproduces both extremes.

In [ ]:

# Search helpers

def make_systematic_G(P):
    """Build a systematic generator matrix G = [I_k | P]."""
    P = np.asarray(P, dtype=float)
    k = P.shape[0]
    return np.hstack([np.eye(k), P])


def project_P_to_train_domain(P, sigma, clip_value=100):
    """Project a candidate P into the integer-valued train data domain."""
    P = np.asarray(P, dtype=float)
    scale = sigma / max(np.std(P), 1e-8)
    projected = np.clip(np.round(P * scale), -clip_value, clip_value).astype(int)
    return projected


def build_G_random(n, k, rng, sigma=1.0, clip_value=100, integerize=True):
    """Dense random generator matrix G = [I_k | P]."""
    r = n - k
    P = rng.normal(0.0, sigma, size=(k, r))
    if integerize:
        P = np.clip(np.round(P), -clip_value, clip_value).astype(int)
    return make_systematic_G(P)


def build_G_generalized_repetition(n, k, scale=1.0, clip_value=100, integerize=True):
    """Generalized repetition style G = [I_k | P] with balanced repeats."""
    r = n - k
    if r <= 0:
        return np.eye(k)
    counts = np.zeros(k, dtype=int)
    cols = []
    for _ in range(r):
        i = int(np.argmin(counts))
        col = np.zeros((k,))
        col[i] = 1.0
        cols.append(col)
        counts[i] += 1
    P = np.stack(cols, axis=1)
    if integerize:
        P = project_P_to_train_domain(P, sigma=max(scale, 1.0), clip_value=clip_value)
    else:
        P = scale * P
    return make_systematic_G(P)


def all_perm_columns_from_base(base):
    """Unique permutations of a base vector, returned as column vectors."""
    from itertools import permutations
    cols = []
    seen = set()
    for p in permutations(base):
        if p in seen:
            continue
        seen.add(p)
        cols.append(np.array(p, dtype=float))
    return cols


def build_G_permutation(n, k, rng, puncture=False, scale=1.0, clip_value=100, integerize=True):
    """Permutation-style construction: choose columns from permutations of one base vector."""
    r = n - k
    if r <= 0:
        return np.eye(k)
    base = np.arange(1, k + 1, dtype=float)
    rng.shuffle(base)
    base = scale * base / np.linalg.norm(base)
    perm_cols = all_perm_columns_from_base(base)
    rng.shuffle(perm_cols)
    if puncture:
        P_cols = perm_cols[:r]
    else:
        P_cols = []
        for i in range(r):
            P_cols.append(perm_cols[i % len(perm_cols)])
    P = np.stack(P_cols, axis=1)
    if integerize:
        P = project_P_to_train_domain(P, sigma=max(scale, 1.0), clip_value=clip_value)
    return make_systematic_G(P)


def mutate_G(G, rng, sigma=1.0, clip_value=100, integerize=True):
    """Gaussian perturbation on P only, preserving systematic form [I|P]."""
    k, n = G.shape
    P = G[:, k:].astype(float).copy()
    P += rng.normal(0.0, sigma, size=P.shape)
    if integerize:
        P = np.clip(np.round(P), -clip_value, clip_value).astype(int)
    return make_systematic_G(P)


def crossover_G(G1, G2, rng, clip_value=100, integerize=True):
    """Column-wise crossover on P blocks."""
    k, n = G1.shape
    r = n - k
    P1 = G1[:, k:]
    P2 = G2[:, k:]
    mask = rng.integers(0, 2, size=r).astype(bool)
    P3 = np.where(mask[np.newaxis, :], P1, P2)
    if integerize:
        P3 = np.clip(np.round(P3), -clip_value, clip_value).astype(int)
    return make_systematic_G(P3)


def evaluate_G_multi_m(G, n, k, target_ms):
    """Return dict m->h_m for requested m values."""
    P = G[:, k:]
    out = {}
    for m in target_ms:
        out[m] = compute_m_height(n, k, m, P)
    return out


def dominates(a, b, target_ms):
    """Pareto dominance for minimization."""
    le_all = all(a[m] <= b[m] for m in target_ms)
    lt_any = any(a[m] < b[m] for m in target_ms)
    return le_all and lt_any


def pareto_front(scored, target_ms):
    """Filter non-dominated candidates."""
    front = []
    for i, ci in enumerate(scored):
        dominated = False
        for j, cj in enumerate(scored):
            if i == j:
                continue
            if dominates(cj["scores"], ci["scores"], target_ms):
                dominated = True
                break
        if not dominated:
            front.append(ci)
    return front


In [ ]:
# Evolutionary search
def search_codes(
    n,
    k,
    target_ms,
    seed=2026,
    pop_size=24,
    generations=8,
    n_mut=24,
    n_cross=16,
    sigma_init=1.0,
    sigma_mut=0.06,
    n_workers=None,
    chunksize=1,
    clip_value=100,
    integerize=True,
    verbose=False,
):
    rng_local = np.random.default_rng(seed)
    target_ms = tuple(sorted(target_ms))
    if n_workers is None:
        n_workers = max(1, min(cpu_count(), pop_size))
    else:
        n_workers = max(1, min(cpu_count(), n_workers))

    # 1) Initialize pool with mixed families
    init_G = []
    n_each = max(2, pop_size // 4)
    for _ in range(n_each):
        init_G.append(build_G_random(n, k, rng_local, sigma=sigma_init, clip_value=clip_value, integerize=integerize))
    for _ in range(n_each):
        init_G.append(build_G_permutation(n, k, rng_local, puncture=False, scale=sigma_init,
                                          clip_value=clip_value, integerize=integerize))
    for _ in range(n_each):
        init_G.append(build_G_permutation(n, k, rng_local, puncture=True, scale=sigma_init,
                                          clip_value=clip_value, integerize=integerize))
    while len(init_G) < pop_size - 1:
        init_G.append(build_G_random(n, k, rng_local, sigma=sigma_init, clip_value=clip_value, integerize=integerize))
    init_G.append(build_G_generalized_repetition(n, k, scale=sigma_init,
                                                 clip_value=clip_value, integerize=integerize))

    def score_pool(pool_G, pool_obj):
        if not pool_G:
            return []
        work_items = [(idx, G, n, k, target_ms) for idx, G in enumerate(pool_G)]
        scored_local = [None] * len(pool_G)
        for idx, scored_entry in pool_obj.imap_unordered(
            _search_score_worker, work_items, chunksize=chunksize,
        ):
            scored_local[idx] = scored_entry
        return scored_local

    with Pool(n_workers) as score_pool_workers:
        scored = score_pool(init_G, score_pool_workers)
        scored.sort(key=lambda x: x["scalar"])
        scored = scored[:pop_size]

        for gen in range(1, generations + 1):
            parents = [x["G"] for x in scored]
            children = []
            for _ in range(n_mut):
                p = parents[rng_local.integers(0, len(parents))]
                children.append(mutate_G(p, rng_local, sigma=sigma_mut,
                                         clip_value=clip_value, integerize=integerize))
            for _ in range(n_cross):
                i1 = rng_local.integers(0, len(parents))
                i2 = rng_local.integers(0, len(parents))
                children.append(crossover_G(parents[i1], parents[i2], rng_local,
                                            clip_value=clip_value, integerize=integerize))
            scored_children = score_pool(children, score_pool_workers)
            scored = scored + scored_children
            scored.sort(key=lambda x: x["scalar"])
            scored = scored[:pop_size]

    front = pareto_front(scored, target_ms)
    front.sort(key=lambda x: x["scalar"])
    return scored, front

In [ ]:
SEARCH_N              = 9
SEARCH_SAMPLE_TARGET  = 10_000
SEARCH_WORKERS        = min(31, cpu_count())
SEARCH_BASE_SEED      = 2026
SEARCH_POP_SIZE       = 300          # 3x larger for deeper search
SEARCH_GENERATIONS    = 10           # 10 gens to find near-optimal codes
SEARCH_MAX_RUNS       = 20           # more runs to collect enough unique samples
SEARCH_CLIP_VALUE     = 100
SEARCH_FRACTION       = 0.40         # 40% search (more budget for low-end coverage)
LOG_BATCH             = 1_000

# Ensure configs and sigma_map are defined even if section 1 wasn't run in this session
_configs = []
for _k in [4, 5, 6]:
    for _m in range(2, SEARCH_N - _k + 1):
        _configs.append((_k, _m))
configs = _configs

sigma_map = {
    (4, 2): 37, (4, 3): 38, (4, 4): 40, (4, 5): 51,
    (5, 2): 35, (5, 3): 37, (5, 4): 46,
    (6, 2): 29, (6, 3): 40,
}

APC_FRACTION = 0.20

# Groups with largest low-end gap get extra search generations
_extra_gens = {
    (4, 2): 5,   # gap: 4x
    (4, 3): 8,   # gap: 13x
    (4, 4): 8,   # gap: 12x
    (4, 5): 10,  # gap: 25x
    (6, 2): 8,   # gap: 18x
}

search_configs = sorted(configs)
base_group_target = SEARCH_SAMPLE_TARGET // len(search_configs)
extra             = SEARCH_SAMPLE_TARGET  % len(search_configs)
group_targets = {
    cfg: base_group_target + (1 if idx < extra else 0)
    for idx, cfg in enumerate(search_configs)
}

print(f"Generating {SEARCH_SAMPLE_TARGET} diverse samples across "
      f"{len(search_configs)} groups (logging every {LOG_BATCH}).")

search_samples   = []
search_m_heights = []
rng_scatter      = np.random.default_rng(seed=9999)
last_logged      = 0
total_start      = time.time()

for cfg_index, (SEARCH_K, SEARCH_M) in enumerate(search_configs):
    target_count = group_targets[(SEARCH_K, SEARCH_M)]
    n_search  = int(target_count * SEARCH_FRACTION)
    n_scatter = target_count - n_search
    sigma     = sigma_map[(SEARCH_K, SEARCH_M)]
    sigma_mut = max(1.0, 0.15 * sigma)
    gens      = SEARCH_GENERATIONS + _extra_gens.get((SEARCH_K, SEARCH_M), 0)

    group_samples = []
    group_heights = []
    seen_group    = set()

    # ── Part A: Evolutionary search (low-m-height enrichment) ─────────────
    group_scored = []
    run_idx      = 0
    while len(group_samples) < n_search:
        if run_idx >= SEARCH_MAX_RUNS:
            break
        run_seed = SEARCH_BASE_SEED + cfg_index * 100 + run_idx
        scored_pool, _ = search_codes(
            n=SEARCH_N, k=SEARCH_K, target_ms=[SEARCH_M],
            seed=run_seed,
            pop_size=SEARCH_POP_SIZE, generations=gens,
            n_mut=int(0.9 * SEARCH_POP_SIZE), n_cross=int(0.5 * SEARCH_POP_SIZE),
            sigma_init=sigma, sigma_mut=sigma_mut,
            n_workers=SEARCH_WORKERS, clip_value=SEARCH_CLIP_VALUE, integerize=True,
        )
        group_scored.extend(scored_pool)
        group_scored.sort(key=lambda x: x["scalar"])
        for cand in group_scored:
            P   = np.clip(np.round(cand['G'][:, SEARCH_K:]),
                          -SEARCH_CLIP_VALUE, SEARCH_CLIP_VALUE).astype(int)
            key = tuple(P.ravel().tolist())
            if key in seen_group:
                continue
            seen_group.add(key)
            group_samples.append([SEARCH_N, SEARCH_K, SEARCH_M, P.copy()])
            group_heights.append(float(cand['scores'][SEARCH_M]))

            total_so_far = len(search_samples) + len(group_samples)
            if total_so_far - last_logged >= LOG_BATCH:
                elapsed = time.time() - total_start
                print(f"  [{total_so_far:>6}/{SEARCH_SAMPLE_TARGET}] "
                      f"{elapsed:.0f}s elapsed  |  "
                      f"k={SEARCH_K}, m={SEARCH_M} (search)")
                sys.stdout.flush()
                last_logged = total_so_far

            if len(group_samples) >= n_search:
                break
        run_idx += 1

    # ── Part B: Random scatter (full distribution) ────────────────────────
    scatter_items    = []
    scatter_rejected = 0
    while len(scatter_items) < n_scatter:
        if rng_scatter.random() < APC_FRACTION:
            P = gen_apc_P(SEARCH_K, SEARCH_N, rng_scatter, sigma)
        else:
            P = gen_gaussian_P(SEARCH_K, SEARCH_N, rng_scatter, sigma)
        d = min_distance(P, SEARCH_K, SEARCH_N)
        if SEARCH_M < d:
            key = tuple(P.ravel().tolist())
            if key not in seen_group:
                seen_group.add(key)
                scatter_items.append([SEARCH_N, SEARCH_K, SEARCH_M, P.copy()])
        else:
            scatter_rejected += 1

    scatter_work = [(i, s[0], s[1], s[2], s[3]) for i, s in enumerate(scatter_items)]
    scatter_h    = [None] * len(scatter_items)
    with Pool(SEARCH_WORKERS) as pool:
        for idx, height in pool.imap_unordered(_worker, scatter_work, chunksize=20):
            scatter_h[idx] = height

            total_so_far = len(search_samples) + len(group_samples) + (idx + 1)
            if total_so_far - last_logged >= LOG_BATCH:
                elapsed = time.time() - total_start
                print(f"  [{total_so_far:>6}/{SEARCH_SAMPLE_TARGET}] "
                      f"{elapsed:.0f}s elapsed  |  "
                      f"k={SEARCH_K}, m={SEARCH_M} (scatter)")
                sys.stdout.flush()
                last_logged = total_so_far

    for i, s in enumerate(scatter_items):
        group_samples.append(s)
        group_heights.append(scatter_h[i])

    search_samples.extend(group_samples)
    search_m_heights.extend(group_heights)

# ── Final summary ─────────────────────────────────────────────────────────
elapsed = time.time() - total_start
h_all   = np.array(search_m_heights)
print(f"\nDone in {elapsed:.1f}s — {len(search_samples)} samples total")
print(f"Height range: [{h_all.min():.2f}, {h_all.max():.2f}]  "
      f"median={np.median(h_all):.2f}")
print(f"\nPer-group summary:")
for grp in search_configs:
    idxs = [i for i, s in enumerate(search_samples) if (s[1], s[2]) == grp]
    h_g  = np.array([search_m_heights[i] for i in idxs])
    print(f"  k={grp[0]},m={grp[1]}: n={len(h_g):5d}  "
          f"min={h_g.min():.2f}  median={np.median(h_g):.2f}  "
          f"max={h_g.max():.2f}")

In [ ]:
# Save search-generated tuples and m-heights
search_out_dir = '/workspace/Homework/Project1'

with open(f'{search_out_dir}/Search_Generated_n_k_m_P', 'wb') as f:
    pickle.dump(search_samples, f)

with open(f'{search_out_dir}/Search_Generated_m-heights', 'wb') as f:
    pickle.dump(search_m_heights, f)

print(f"Saved {len(search_samples)} search samples to Search_Generated_n_k_m_P")
print(f"Saved {len(search_m_heights)} search m-heights to Search_Generated_m-heights")

# Verify by reloading
with open(f'{search_out_dir}/Search_Generated_n_k_m_P', 'rb') as f:
    check_search_data = pickle.load(f)
with open(f'{search_out_dir}/Search_Generated_m-heights', 'rb') as f:
    check_search_heights = pickle.load(f)

print(f"Verification: {len(check_search_data)} samples, {len(check_search_heights)} heights")
if len(check_search_heights) > 0:
    print(f"Heights range: [{min(check_search_heights):.6f}, {max(check_search_heights):.6f}]")
